# RSNA input audit

This notebook contains the reviewed Python audit sources and uses the standard library only. Attach competition inputs, keep Internet Off, and Run All. No CLI authentication or package installation is needed.

The default run discovers input structure with bounded traversal. Generate a CSV-inspection variant locally with `python scripts/build_audit_notebook.py --inspect-csv --output notebooks/kaggle_audit_csv.ipynb`; import that complete file instead of adding cells. This notebook does not decode DICOMs or train models. Synthetic checks do not verify competition access, actual data, or model performance.


In [ ]:
# Reviewed source bundle: no filesystem extraction or network access.
import hashlib
import importlib.abc
import importlib.util
import json
import sys

_RSNA_BUNDLE = json.loads('{"sha256":{"__init__.py":"f7c27acdb5e30e9734b6fbb6d55809ec8bcfef85c15cbac858331d23941b994e","audit.py":"13bcb1acc910af81000805e1e32d139475fc7b97cbdcf5cd4d8d6320731cfbd5","cli.py":"64f2631094bcc8aab879b2ce4471731aa3fd7d39add1bbea5f019221c73ceff6","config.py":"6dbb61af824770c3ac0dca64636762f14f9cc71e3c3bd0193b200211996347cd","contracts.py":"34d3892dd8fe63dde0f5494815240b008082406b94e6fff0d16528b45c3fda05","inputs.py":"48fa4fe5c0c5f183e6975113cfd889ae2c8c9139e9b98b944dc3708e362f5d8e","metrics.py":"4121fb00bd7e2666b3f0885d32d1430d7e102d9ee8d70d4f7bf8d27dbea2f774","submission.py":"9dc8941373de54bfa2ebe037e485d0060baea653bec6c981aa6a447c7673d5b0"},"sources":{"__init__":"\\"\\"\\"Read-only audit software; DICOM support and model performance require evidence.\\"\\"\\"\\r\\n\\r\\n__version__ = \\"0.1.0\\"\\r\\n","audit":"\\"\\"\\"Initial preflight contracts only; never a Phase B or training approval.\\"\\"\\"\\r\\n\\r\\nfrom datetime import datetime, timezone\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\nimport platform\\r\\nimport time\\r\\nimport uuid\\r\\n\\r\\nfrom .config import AuditConfig\\r\\nfrom .contracts import ContractError, TARGETS, audit_series, audit_studies\\r\\nfrom .inputs import InputError, MANIFEST_NAMES, discover_inputs, read_csv_manifest, safe_absolute\\r\\nfrom .submission import validate_submission\\r\\n\\r\\n\\r\\ndef _summarize(contract: dict) -> dict:\\r\\n    # Keep outputs compact; no full identifier tables, report text, or pixels.\\r\\n    return {key: value for key, value in contract.items() if key not in (\\"study_ids\\", \\"series_ids\\")}\\r\\n\\r\\n\\r\\ndef _csv_contracts(root: str, config: AuditConfig, progress: dict) -> dict:\\r\\n    tables = {}\\r\\n    columns = {}\\r\\n    hashes = {}\\r\\n    for name in MANIFEST_NAMES:\\r\\n        progress[\\"attempted_files\\"].append(name)\\r\\n        columns[name], tables[name], hashes[name] = read_csv_manifest(Path(root) / name, config)\\r\\n        progress[\\"parsed_files\\"].append(name)\\r\\n        progress[\\"sha256\\"][name] = hashes[name]\\r\\n    train = audit_studies(tables[\\"train.csv\\"], \\"train\\")\\r\\n    test = audit_studies(tables[\\"test.csv\\"], \\"test\\")\\r\\n    train_ids, test_ids = set(train[\\"study_ids\\"]), set(test[\\"study_ids\\"])\\r\\n    overlap = train_ids & test_ids\\r\\n    if overlap:\\r\\n        raise ContractError(\\"Train/test study identifiers overlap\\")\\r\\n    train_series = audit_series(tables[\\"train_series.csv\\"], train_ids)\\r\\n    test_series = audit_series(tables[\\"test_series.csv\\"], test_ids)\\r\\n    series_overlap = set(train_series[\\"series_ids\\"]) & set(test_series[\\"series_ids\\"])\\r\\n    if series_overlap:\\r\\n        raise ContractError(\\"Train/test series identifiers overlap\\")\\r\\n    missing_train = train_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"train_series.csv\\"]}\\r\\n    missing_test = test_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"test_series.csv\\"]}\\r\\n    if missing_train or missing_test:\\r\\n        raise ContractError(\\"Study manifest includes studies without a series reference\\")\\r\\n    sample = []\\r\\n    for row in tables[\\"sample_submission.csv\\"]:\\r\\n        converted = dict(row)\\r\\n        for target in TARGETS:\\r\\n            if target not in converted:\\r\\n                raise ContractError(\\"Sample submission is missing target columns\\")\\r\\n            try:\\r\\n                converted[target] = float(converted[target])\\r\\n            except (TypeError, ValueError) as error:\\r\\n                raise ContractError(\\"Sample submission scores must be numeric\\") from error\\r\\n        sample.append(converted)\\r\\n    submission = validate_submission(sample, test_ids, columns=columns[\\"sample_submission.csv\\"])\\r\\n    valid = all(result[\\"status\\"] == \\"PASS\\" for result in (train, test, train_series, test_series))\\r\\n    return {\\r\\n        \\"status\\": \\"VALID\\" if valid else \\"INVALID\\", \\"columns\\": columns,\\r\\n        \\"sha256\\": hashes, \\"train\\": _summarize(train), \\"test\\": _summarize(test),\\r\\n        \\"train_series\\": _summarize(train_series), \\"test_series\\": _summarize(test_series),\\r\\n        \\"sample_submission\\": submission, \\"train_test_study_overlap_count\\": 0,\\r\\n        \\"train_test_series_overlap_count\\": 0,\\r\\n        \\"scope\\": \\"CSV contracts only; no patient, report-quality, geometry or image-integrity audit\\",\\r\\n    }\\r\\n\\r\\n\\r\\ndef run_preflight(config: AuditConfig, source_identity: dict | None = None) -> dict:\\r\\n    if not isinstance(config, AuditConfig):\\r\\n        raise TypeError(\\"run_preflight requires validated AuditConfig\\")\\r\\n    started = time.perf_counter()\\r\\n    discovery = discover_inputs(config)\\r\\n    csv_contracts = {\\"status\\": \\"NOT_REQUESTED\\"}\\r\\n    status = discovery[\\"status\\"]\\r\\n    if config.inspect_csv:\\r\\n        csv_contracts = {\\"status\\": \\"BLOCKED_BY_DISCOVERY\\"}\\r\\n        if discovery[\\"selected_root\\"]:\\r\\n            progress = {\\"attempted_files\\": [], \\"parsed_files\\": [], \\"sha256\\": {}}\\r\\n            discovery[\\"csv_read_attempted\\"] = True\\r\\n            discovery[\\"csv_read_completed\\"] = False\\r\\n            discovery[\\"csv_preflight_completed\\"] = False\\r\\n            try:\\r\\n                csv_contracts = _csv_contracts(discovery[\\"selected_root\\"], config, progress)\\r\\n                status = \\"CONTRACTS_VALID\\" if csv_contracts[\\"status\\"] == \\"VALID\\" else \\"CONTRACTS_INVALID\\"\\r\\n                discovery[\\"csv_preflight_completed\\"] = True\\r\\n            except (InputError, ContractError, OSError) as error:\\r\\n                # Exceptions contain only static contract descriptions; no raw reports.\\r\\n                csv_contracts = {\\"status\\": \\"INVALID\\", \\"error_type\\": type(error).__name__,\\r\\n                                 \\"reason\\": str(error) if isinstance(error, (InputError, ContractError)) else \\"Filesystem read failed\\"}\\r\\n                status = \\"CONTRACTS_INVALID\\"\\r\\n            discovery[\\"csv_read_completed\\"] = len(progress[\\"parsed_files\\"]) == len(MANIFEST_NAMES)\\r\\n            # A parse failure may still have consumed bytes. None means unknown,\\r\\n            # while successful parsed files establish that contents were read.\\r\\n            discovery[\\"csv_contents_read\\"] = True if progress[\\"parsed_files\\"] else None\\r\\n            discovery[\\"csv_read_progress\\"] = progress\\r\\n    return {\\r\\n        \\"schema\\": \\"rsna-knee-preflight-v1\\", \\"phase\\": \\"INITIAL_SCAFFOLD\\", \\"status\\": status,\\r\\n        \\"created_utc\\": datetime.now(timezone.utc).isoformat(), \\"configuration\\": config.to_dict(),\\r\\n        \\"source_identity\\": source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"},\\r\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"platform\\": platform.system(), \\"gpu_used\\": False},\\r\\n        \\"discovery\\": discovery, \\"csv_contracts\\": csv_contracts,\\r\\n        \\"phase_b_status\\": \\"NOT_RUN\\", \\"training_ready\\": False, \\"phase1_pass\\": False,\\r\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"observed_study_count\\": None},\\r\\n        \\"model_metrics\\": None,\\r\\n        \\"unassessed\\": [\\"patient grouping\\", \\"report quality and languages\\", \\"DICOM UID inventory\\",\\r\\n                       \\"physical geometry\\", \\"transfer syntaxes and decoding\\", \\"gold provenance\\",\\r\\n                       \\"validation split viability\\", \\"Kaggle account access\\", \\"model and inference runtime\\"],\\r\\n        \\"duration_before_report_write_seconds\\": time.perf_counter() - started,\\r\\n    }\\r\\n\\r\\n\\r\\ndef write_report(report: dict, config: AuditConfig) -> Path:\\r\\n    output = safe_absolute(config.output_dir)\\r\\n    source = safe_absolute(config.input_base)\\r\\n    if output.is_relative_to(source):\\r\\n        raise InputError(\\"Report destination must be outside input_base\\")\\r\\n    payload = json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + \\"\\\\n\\"\\r\\n    if len(payload.encode(\\"utf-8\\")) > config.max_report_bytes:\\r\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\r\\n    output.mkdir(parents=True, exist_ok=True)\\r\\n    stamp = datetime.now(timezone.utc).strftime(\\"%Y%m%dT%H%M%S%fZ\\")\\r\\n    path = output / f\\"preflight_{stamp}_{uuid.uuid4().hex[:8]}.json\\"\\r\\n    with path.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as stream:\\r\\n        stream.write(payload)\\r\\n    return path\\r\\n","cli":"\\"\\"\\"Local entry point with no authenticated services or training commands.\\"\\"\\"\\r\\n\\r\\nimport argparse\\r\\nimport hashlib\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\nimport sys\\r\\n\\r\\nfrom .audit import run_preflight, write_report\\r\\nfrom .config import AuditConfig, ConfigError, load_config\\r\\nfrom .inputs import InputError\\r\\n\\r\\n\\r\\ndef _source_identity() -> dict:\\r\\n    package = Path(__file__).parent\\r\\n    names = (\\"__init__\\", \\"config\\", \\"inputs\\", \\"contracts\\", \\"metrics\\", \\"submission\\", \\"audit\\", \\"cli\\")\\r\\n    return {\\"modules\\": {name: hashlib.sha256((package / (name + \\".py\\")).read_bytes()).hexdigest() for name in names}}\\r\\n\\r\\n\\r\\ndef main(argv=None) -> int:\\r\\n    parser = argparse.ArgumentParser(description=\\"Read-only RSNA input/CSV preflight; no Phase B execution or training\\")\\r\\n    parser.add_argument(\\"--config\\", type=Path)\\r\\n    parser.add_argument(\\"--input-base\\")\\r\\n    parser.add_argument(\\"--data-root\\")\\r\\n    parser.add_argument(\\"--output-dir\\")\\r\\n    parser.add_argument(\\"--inspect-csv\\", action=\\"store_true\\", default=None,\\r\\n                        help=\\"Check bounded CSV contracts; source data is not modified\\")\\r\\n    args = parser.parse_args(argv)\\r\\n    try:\\r\\n        config = load_config(args.config) if args.config else AuditConfig()\\r\\n        overrides = {name: getattr(args, name) for name in (\\"input_base\\", \\"data_root\\", \\"output_dir\\", \\"inspect_csv\\") if getattr(args, name) is not None}\\r\\n        config = config.with_overrides(**overrides)\\r\\n        report = run_preflight(config, source_identity=_source_identity())\\r\\n        path = write_report(report, config)\\r\\n    except (ConfigError, InputError, OSError, ValueError) as error:\\r\\n        message = str(error) if isinstance(error, (ConfigError, InputError)) else \\"Preflight failed; check configuration and filesystem permissions\\"\\r\\n        print(json.dumps({\\"status\\": \\"ERROR\\", \\"error_type\\": type(error).__name__, \\"reason\\": message}), file=sys.stderr)\\r\\n        return 2\\r\\n    print(json.dumps({\\"status\\": report[\\"status\\"], \\"report_path\\": str(path),\\r\\n                      \\"phase_b_status\\": report[\\"phase_b_status\\"], \\"training_ready\\": report[\\"training_ready\\"]}, indent=2))\\r\\n    return 0 if report[\\"status\\"] in (\\"MANIFESTS_LOCATED\\", \\"CONTRACTS_VALID\\") else 2\\r\\n","config":"\\"\\"\\"Small immutable configuration; no credential or environment discovery.\\"\\"\\"\\r\\n\\r\\nfrom dataclasses import asdict, dataclass, fields, replace\\r\\nimport json\\r\\nimport os\\r\\nfrom pathlib import Path\\r\\nimport stat\\r\\nfrom typing import Mapping\\r\\n\\r\\n\\r\\nclass ConfigError(ValueError):\\r\\n    \\"\\"\\"Configuration cannot safely represent the requested preflight.\\"\\"\\"\\r\\n\\r\\n\\r\\n@dataclass(frozen=True)\\r\\nclass AuditConfig:\\r\\n    input_base: str = \\"data\\"\\r\\n    output_dir: str = \\"artifacts\\"\\r\\n    data_root: str | None = None\\r\\n    inspect_csv: bool = False\\r\\n    max_depth: int = 2\\r\\n    max_directories: int = 128\\r\\n    max_entries: int = 2048\\r\\n    max_file_bytes: int = 8 * 1024 * 1024\\r\\n    max_csv_rows: int = 100000\\r\\n    max_report_bytes: int = 1024 * 1024\\r\\n\\r\\n    def __post_init__(self):\\r\\n        for name in (\\"input_base\\", \\"output_dir\\", \\"data_root\\"):\\r\\n            value = getattr(self, name)\\r\\n            if value is None and name == \\"data_root\\":\\r\\n                continue\\r\\n            if not isinstance(value, str) or not value.strip() or value != value.strip():\\r\\n                raise ConfigError(f\\"{name} must be a nonempty unpadded path string\\")\\r\\n            if any(ord(character) < 32 for character in value):\\r\\n                raise ConfigError(f\\"{name} contains a control character\\")\\r\\n        if type(self.inspect_csv) is not bool:\\r\\n            raise ConfigError(\\"inspect_csv must be a JSON boolean\\")\\r\\n        limits = {\\r\\n            \\"max_depth\\": (0, 4), \\"max_directories\\": (1, 1024),\\r\\n            \\"max_entries\\": (1, 65536), \\"max_file_bytes\\": (1, 64 * 1024 * 1024),\\r\\n            \\"max_csv_rows\\": (1, 1000000), \\"max_report_bytes\\": (1, 4 * 1024 * 1024),\\r\\n        }\\r\\n        for name, (lower, upper) in limits.items():\\r\\n            value = getattr(self, name)\\r\\n            if type(value) is not int or not lower <= value <= upper:\\r\\n                raise ConfigError(f\\"{name} must be an integer in [{lower}, {upper}]\\")\\r\\n\\r\\n    @classmethod\\r\\n    def from_dict(cls, values: Mapping):\\r\\n        if not isinstance(values, Mapping):\\r\\n            raise ConfigError(\\"Configuration must be a JSON object\\")\\r\\n        unknown = set(values) - {field.name for field in fields(cls)}\\r\\n        if unknown:\\r\\n            raise ConfigError(\\"Unknown configuration fields: \\" + \\", \\".join(sorted(str(key) for key in unknown)))\\r\\n        return cls(**dict(values))\\r\\n\\r\\n    def to_dict(self) -> dict:\\r\\n        return asdict(self)\\r\\n\\r\\n    def with_overrides(self, **values):\\r\\n        try:\\r\\n            return replace(self, **values)\\r\\n        except TypeError as error:\\r\\n            raise ConfigError(\\"Invalid configuration override\\") from error\\r\\n\\r\\n\\r\\ndef _read_config_values(path: str | Path) -> dict:\\r\\n    \\"\\"\\"Shared bounded JSON reader; never discover environment or credentials.\\"\\"\\"\\r\\n    def unique_pairs(pairs):\\r\\n        values = {}\\r\\n        for key, value in pairs:\\r\\n            if key in values:\\r\\n                raise ConfigError(f\\"Duplicate configuration field: {key}\\")\\r\\n            values[key] = value\\r\\n        return values\\r\\n\\r\\n    source = Path(os.path.abspath(path))\\r\\n    protected = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\r\\n    if source.suffix.casefold() != \\".json\\" or any(part.casefold() in protected or part.casefold().startswith(\\".env\\") for part in source.parts):\\r\\n        raise ConfigError(\\"Configuration must be a project JSON file outside protected paths\\")\\r\\n    for ancestor in (source, *source.parents):\\r\\n        try:\\r\\n            info = ancestor.lstat()\\r\\n        except FileNotFoundError:\\r\\n            continue\\r\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n            raise ConfigError(\\"Configuration cannot use symbolic links or reparse paths\\")\\r\\n    if not stat.S_ISREG(source.lstat().st_mode):\\r\\n        raise ConfigError(\\"Configuration must be a regular project JSON file\\")\\r\\n    with source.open(\\"rb\\") as stream:\\r\\n        raw = stream.read(64 * 1024 + 1)\\r\\n    if len(raw) > 64 * 1024:\\r\\n        raise ConfigError(\\"Configuration exceeds the 64 KiB budget\\")\\r\\n    try:\\r\\n        values = json.loads(raw.decode(\\"utf-8-sig\\"), object_pairs_hook=unique_pairs)\\r\\n    except (json.JSONDecodeError, UnicodeError) as error:\\r\\n        raise ConfigError(\\"Configuration must be valid UTF-8 JSON\\") from error\\r\\n    return values\\r\\n\\r\\n\\r\\ndef load_config(path: str | Path) -> AuditConfig:\\r\\n    return AuditConfig.from_dict(_read_config_values(path))\\r\\n","contracts":"\\"\\"\\"Small, explicit contracts for competition identifiers and supervision.\\r\\n\\r\\nUnknown labels never become negatives. These functions inspect tabular values;\\r\\nthey neither decode images nor infer labels from reports.\\r\\n\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport math\\r\\nimport unicodedata\\r\\nfrom collections import Counter\\r\\nfrom collections.abc import Mapping\\r\\nfrom decimal import Decimal\\r\\nfrom numbers import Real\\r\\nfrom typing import Any\\r\\n\\r\\nTARGETS = (\\r\\n    \\"ACL\\", \\"MCL\\", \\"Medial Meniscus\\", \\"Lateral Meniscus\\", \\"Medial OA\\",\\r\\n    \\"Lateral OA\\", \\"PF OA\\", \\"Effusion\\", \\"Synovitis\\", \\"Baker\'s\\",\\r\\n    \\"Contusion\\", \\"Fracture\\",\\r\\n)\\r\\nSTUDY_ID = \\"StudyInstanceUID\\"\\r\\nSERIES_ID = \\"SeriesInstanceUID\\"\\r\\nSERIES_COLUMNS = (\\r\\n    STUDY_ID, SERIES_ID, \\"Anatomical_Plane\\", \\"Fluid_Sensitive\\", \\"Fat_Suppression\\",\\r\\n)\\r\\n\\r\\n\\r\\nclass ContractError(ValueError):\\r\\n    \\"\\"\\"A table violates a declared data contract.\\"\\"\\"\\r\\n\\r\\n\\r\\ndef validate_identifier(value: str) -> str:\\r\\n    \\"\\"\\"Return an unchanged safe identifier without imposing a numeric UID format.\\"\\"\\"\\r\\n    if not isinstance(value, str) or not value or value != value.strip():\\r\\n        raise ContractError(\\"identifier must be a nonempty, unpadded string\\")\\r\\n    if value in {\\".\\", \\"..\\"} or any(char in value for char in \\"/\\\\\\\\:\\"):\\r\\n        raise ContractError(\\"identifier must be a single safe path component\\")\\r\\n    if any(unicodedata.category(char) in {\\"Cc\\", \\"Cf\\"} for char in value):\\r\\n        raise ContractError(\\"identifier contains a control character\\")\\r\\n    return value\\r\\n\\r\\n\\r\\ndef _numeric(value: Any) -> bool:\\r\\n    return not isinstance(value, bool) and isinstance(value, (Real, Decimal))\\r\\n\\r\\n\\r\\ndef parse_label(value: Any) -> tuple[float | None, str]:\\r\\n    \\"\\"\\"Classify an observed binary label, an unknown cell, or an invalid cell.\\r\\n\\r\\n    Accepted text is deliberately limited to 0, 1, 0.0 and 1.0. A text token\\r\\n    such as \'NaN\' is invalid; an actual floating NaN represents a missing cell.\\r\\n    \\"\\"\\"\\r\\n    if value is None:\\r\\n        return None, \\"unknown\\"\\r\\n    if isinstance(value, str):\\r\\n        token = value.strip()\\r\\n        if not token:\\r\\n            return None, \\"unknown\\"\\r\\n        if token in {\\"0\\", \\"0.0\\", \\"1\\", \\"1.0\\"}:\\r\\n            return float(token), \\"observed\\"\\r\\n        return None, \\"invalid\\"\\r\\n    if isinstance(value, Real) and not isinstance(value, bool):\\r\\n        try:\\r\\n            if math.isnan(float(value)):\\r\\n                return None, \\"unknown\\"\\r\\n        except (OverflowError, ValueError):\\r\\n            return None, \\"invalid\\"\\r\\n    if _numeric(value):\\r\\n        try:\\r\\n            numeric = float(value)\\r\\n        except (OverflowError, ValueError):\\r\\n            return None, \\"invalid\\"\\r\\n        if math.isfinite(numeric) and numeric in (0.0, 1.0):\\r\\n            return numeric, \\"observed\\"\\r\\n    return None, \\"invalid\\"\\r\\n\\r\\n\\r\\ndef _required(row: Mapping[str, Any], required: tuple[str, ...], index: int) -> None:\\r\\n    if not isinstance(row, Mapping):\\r\\n        raise ContractError(f\\"row {index} must be a mapping\\")\\r\\n    missing = [key for key in required if key not in row]\\r\\n    if missing:\\r\\n        raise ContractError(f\\"row {index} is missing required columns: {\', \'.join(missing)}\\")\\r\\n\\r\\n\\r\\ndef label_vector(row: Mapping[str, Any]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Return twelve ordered values and an explicit observation mask.\\r\\n\\r\\n    Unknown and invalid cells retain None; the state list records the\\r\\n    distinction. Only verified binary observations receive a True mask.\\r\\n    \\"\\"\\"\\r\\n    _required(row, TARGETS, 0)\\r\\n    parsed = [parse_label(row[target]) for target in TARGETS]\\r\\n    return {\\r\\n        \\"targets\\": list(TARGETS),\\r\\n        \\"values\\": [value for value, _ in parsed],\\r\\n        \\"observed_mask\\": [state == \\"observed\\" for _, state in parsed],\\r\\n        \\"states\\": [state for _, state in parsed],\\r\\n    }\\r\\n\\r\\n\\r\\ndef audit_studies(rows: list[dict[str, Any]], split: str) -> dict[str, Any]:\\r\\n    \\"\\"\\"Audit study identities and, for training, labels and report coverage.\\"\\"\\"\\r\\n    if split not in {\\"train\\", \\"test\\"}:\\r\\n        raise ContractError(\\"split must be train or test\\")\\r\\n    if not rows:\\r\\n        raise ContractError(\\"study table must contain at least one row\\")\\r\\n    required = (STUDY_ID, *TARGETS, \\"Report\\") if split == \\"train\\" else (STUDY_ID,)\\r\\n    study_ids: list[str] = []\\r\\n    seen: set[str] = set()\\r\\n    statistics = {\\r\\n        target: {\\"observed\\": 0, \\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0, \\"invalid\\": 0}\\r\\n        for target in TARGETS\\r\\n    }\\r\\n    label_states: list[dict[str, tuple[float | None, str]]] = []\\r\\n    report_summary = {\\"present\\": 0, \\"empty_or_missing\\": 0, \\"invalid\\": 0}\\r\\n    for index, row in enumerate(rows):\\r\\n        _required(row, required, index)\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in seen:\\r\\n            raise ContractError(f\\"duplicate study identifier at row {index}\\")\\r\\n        seen.add(uid)\\r\\n        study_ids.append(uid)\\r\\n        if split == \\"test\\":\\r\\n            continue\\r\\n        states = {target: parse_label(row[target]) for target in TARGETS}\\r\\n        label_states.append(states)\\r\\n        for target, (label, state) in states.items():\\r\\n            statistics[target][state] += 1\\r\\n            if state == \\"observed\\":\\r\\n                statistics[target][\\"positive\\" if label == 1.0 else \\"negative\\"] += 1\\r\\n        report = row[\\"Report\\"]\\r\\n        if report is None or (isinstance(report, str) and not report.strip()):\\r\\n            report_summary[\\"empty_or_missing\\"] += 1\\r\\n        elif isinstance(report, str):\\r\\n            report_summary[\\"present\\"] += 1\\r\\n        else:\\r\\n            report_summary[\\"invalid\\"] += 1\\r\\n    result: dict[str, Any] = {\\r\\n        \\"status\\": \\"PASS\\", \\"split\\": split, \\"row_count\\": len(rows),\\r\\n        \\"unique_study_count\\": len(seen), \\"study_ids\\": study_ids,\\r\\n    }\\r\\n    if split == \\"test\\":\\r\\n        return result\\r\\n    for stats in statistics.values():\\r\\n        observed = stats[\\"observed\\"]\\r\\n        stats[\\"prevalence\\"] = stats[\\"positive\\"] / observed if observed else None\\r\\n        stats[\\"observation_coverage\\"] = observed / len(rows)\\r\\n    cooccurrence = []\\r\\n    for first_index, first in enumerate(TARGETS):\\r\\n        for second in TARGETS[first_index + 1:]:\\r\\n            jointly_observed = both_positive = 0\\r\\n            for states in label_states:\\r\\n                first_label, first_state = states[first]\\r\\n                second_label, second_state = states[second]\\r\\n                if first_state == second_state == \\"observed\\":\\r\\n                    jointly_observed += 1\\r\\n                    both_positive += first_label == second_label == 1.0\\r\\n            cooccurrence.append({\\r\\n                \\"targets\\": [first, second], \\"jointly_observed\\": jointly_observed,\\r\\n                \\"both_positive\\": both_positive,\\r\\n                \\"both_positive_fraction\\": both_positive / jointly_observed if jointly_observed else None,\\r\\n            })\\r\\n    result.update({\\r\\n        \\"target_statistics\\": statistics, \\"report_summary\\": report_summary,\\r\\n        \\"observation_coverage\\": sum(stats[\\"observed\\"] for stats in statistics.values()) / (len(rows) * len(TARGETS)),\\r\\n        \\"jointly_observed_cooccurrence\\": cooccurrence,\\r\\n    })\\r\\n    if any(stats[\\"invalid\\"] for stats in statistics.values()) or report_summary[\\"invalid\\"]:\\r\\n        result[\\"status\\"] = \\"INVALID\\"\\r\\n    return result\\r\\n\\r\\n\\r\\ndef _metadata_state(value: Any) -> str:\\r\\n    if value is None or (isinstance(value, str) and not value.strip()):\\r\\n        return \\"unknown\\"\\r\\n    if isinstance(value, float):\\r\\n        if math.isnan(float(value)):\\r\\n            return \\"unknown\\"\\r\\n        if not math.isfinite(float(value)):\\r\\n            return \\"invalid\\"\\r\\n    if isinstance(value, (str, bool, int, float)):\\r\\n        return \\"observed\\"\\r\\n    return \\"invalid\\"\\r\\n\\r\\n\\r\\ndef audit_series(rows: list[dict[str, Any]], study_ids: set[str]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Check series uniqueness, study references and exact metadata coverage.\\r\\n\\r\\n    Category vocabularies must be established from real data. No plane or flag\\r\\n    value is coerced here; typed value counts retain the observed representation.\\r\\n    \\"\\"\\"\\r\\n    if not rows:\\r\\n        raise ContractError(\\"series table must contain at least one row\\")\\r\\n    validated_studies = {validate_identifier(uid) for uid in study_ids}\\r\\n    seen: set[str] = set()\\r\\n    referenced: set[str] = set()\\r\\n    metadata_fields = SERIES_COLUMNS[2:]\\r\\n    statistics = {field: {\\"observed\\": 0, \\"unknown\\": 0, \\"invalid\\": 0} for field in metadata_fields}\\r\\n    counts: dict[str, Counter[tuple[str, Any]]] = {field: Counter() for field in metadata_fields}\\r\\n    series_ids: list[str] = []\\r\\n    for index, row in enumerate(rows):\\r\\n        _required(row, SERIES_COLUMNS, index)\\r\\n        study = validate_identifier(row[STUDY_ID])\\r\\n        series = validate_identifier(row[SERIES_ID])\\r\\n        if study not in validated_studies:\\r\\n            raise ContractError(f\\"series row {index} references an absent study\\")\\r\\n        if series in seen:\\r\\n            raise ContractError(f\\"duplicate series identifier at row {index}\\")\\r\\n        seen.add(series)\\r\\n        series_ids.append(series)\\r\\n        referenced.add(study)\\r\\n        for field in metadata_fields:\\r\\n            value = row[field]\\r\\n            state = _metadata_state(value)\\r\\n            statistics[field][state] += 1\\r\\n            if state == \\"observed\\":\\r\\n                counts[field][(type(value).__name__, value)] += 1\\r\\n    for field in metadata_fields:\\r\\n        statistics[field][\\"value_counts\\"] = [\\r\\n            {\\"type\\": value_type, \\"value\\": value, \\"count\\": count}\\r\\n            for (value_type, value), count in sorted(counts[field].items(), key=lambda item: (item[0][0], str(item[0][1])))\\r\\n        ]\\r\\n    return {\\r\\n        \\"status\\": \\"INVALID\\" if any(stats[\\"invalid\\"] for stats in statistics.values()) else \\"PASS\\",\\r\\n        \\"row_count\\": len(rows), \\"unique_series_count\\": len(seen),\\r\\n        \\"study_count_with_series\\": len(referenced), \\"study_count_without_series\\": len(validated_studies - referenced),\\r\\n        \\"series_ids\\": series_ids, \\"metadata_statistics\\": statistics,\\r\\n    }\\r\\n","inputs":"\\"\\"\\"Bounded read-only discovery and CSV reads with explicit incomplete states.\\"\\"\\"\\r\\n\\r\\nimport csv\\r\\nfrom datetime import datetime, timezone\\r\\nimport hashlib\\r\\nimport io\\r\\nimport os\\r\\nfrom pathlib import Path\\r\\nimport stat\\r\\n\\r\\nfrom .config import AuditConfig\\r\\n\\r\\n\\r\\nMANIFEST_NAMES = (\\"train.csv\\", \\"train_series.csv\\", \\"test.csv\\", \\"test_series.csv\\", \\"sample_submission.csv\\")\\r\\nIMAGE_DIRECTORIES = {\\"train_series\\", \\"test_series\\", \\"train_images\\", \\"test_images\\"}\\r\\nPROTECTED_COMPONENTS = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\".venv\\", \\"venv\\", \\"node_modules\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\r\\n\\r\\n\\r\\nclass InputError(ValueError):\\r\\n    \\"\\"\\"Input contracts or safe-path requirements were not satisfied.\\"\\"\\"\\r\\n\\r\\n\\r\\ndef safe_absolute(path: str | Path) -> Path:\\r\\n    \\"\\"\\"Reject links/reparse ancestors rather than silently resolving through them.\\"\\"\\"\\r\\n    absolute = Path(os.path.abspath(path))\\r\\n    if any(part.casefold() in PROTECTED_COMPONENTS or part.casefold().startswith(\\".env\\") for part in absolute.parts):\\r\\n        raise InputError(\\"Protected directory is not a permitted input or output path\\")\\r\\n    for ancestor in (absolute, *absolute.parents):\\r\\n        try:\\r\\n            info = ancestor.lstat()\\r\\n        except FileNotFoundError:\\r\\n            continue\\r\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n            raise InputError(\\"Symbolic links and reparse points are not permitted\\")\\r\\n    return absolute\\r\\n\\r\\n\\r\\ndef discover_inputs(config: AuditConfig) -> dict:\\r\\n    result = {\\r\\n        \\"status\\": \\"INPUTS_NOT_LOCATED\\", \\"input_base\\": None, \\"selected_root\\": None,\\r\\n        \\"directories_checked\\": 0, \\"entries_checked\\": 0, \\"scan_complete\\": True,\\r\\n        \\"limit_reached\\": False, \\"errors\\": [], \\"error_count\\": 0,\\r\\n        \\"skipped_links\\": [], \\"skipped_link_count\\": 0, \\"candidates\\": [],\\r\\n        \\"csv_contents_read\\": False, \\"dicom_contents_read\\": False,\\r\\n        \\"dataset_identity_verified\\": False,\\r\\n    }\\r\\n\\r\\n    def issue(path, error):\\r\\n        result[\\"error_count\\"] += 1\\r\\n        if len(result[\\"errors\\"]) < 32:\\r\\n            result[\\"errors\\"].append({\\"path\\": str(path), \\"type\\": type(error).__name__, \\"errno\\": getattr(error, \\"errno\\", None)})\\r\\n\\r\\n    try:\\r\\n        base = safe_absolute(config.input_base)\\r\\n        result[\\"input_base\\"] = str(base)\\r\\n        start = safe_absolute(config.data_root) if config.data_root else base\\r\\n        if not start.is_relative_to(base):\\r\\n            raise InputError(\\"Explicit data_root must be within input_base\\")\\r\\n        info = start.lstat()\\r\\n        if not stat.S_ISDIR(info.st_mode):\\r\\n            raise InputError(\\"Input root is not a directory\\")\\r\\n    except FileNotFoundError:\\r\\n        result[\\"status\\"] = \\"INPUT_ROOT_MISSING\\" if config.data_root else \\"INPUT_BASE_MISSING\\"\\r\\n        return result\\r\\n    except (OSError, InputError) as error:\\r\\n        issue(config.input_base, error)\\r\\n        result.update(status=\\"INPUT_PATH_INVALID\\", scan_complete=False)\\r\\n        return result\\r\\n\\r\\n    def checked_stat(path):\\r\\n        try:\\r\\n            info = path.lstat()\\r\\n            if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n                result[\\"skipped_link_count\\"] += 1\\r\\n                if len(result[\\"skipped_links\\"]) < 32:\\r\\n                    result[\\"skipped_links\\"].append(str(path))\\r\\n                return None\\r\\n            if not path.resolve().is_relative_to(base):\\r\\n                raise InputError(\\"Discovered path escaped input_base\\")\\r\\n            return info\\r\\n        except FileNotFoundError:\\r\\n            return None\\r\\n        except (OSError, InputError) as error:\\r\\n            issue(path, error)\\r\\n            return None\\r\\n\\r\\n    pending = [(start, 0)]\\r\\n    cursor = 0\\r\\n    while cursor < len(pending) and result[\\"directories_checked\\"] < config.max_directories:\\r\\n        directory, depth = pending[cursor]\\r\\n        cursor += 1\\r\\n        result[\\"directories_checked\\"] += 1\\r\\n        files = []\\r\\n        for name in MANIFEST_NAMES:\\r\\n            path = directory / name\\r\\n            info = checked_stat(path)\\r\\n            if info is not None and stat.S_ISREG(info.st_mode):\\r\\n                files.append({\\"name\\": name, \\"path\\": str(path), \\"size_bytes\\": info.st_size,\\r\\n                              \\"modified_utc\\": datetime.fromtimestamp(info.st_mtime, timezone.utc).isoformat(), \\"sha256\\": None})\\r\\n        if files:\\r\\n            present = {item[\\"name\\"] for item in files}\\r\\n            images = {}\\r\\n            for name in sorted(IMAGE_DIRECTORIES):\\r\\n                path = directory / name\\r\\n                info = checked_stat(path)\\r\\n                if info is not None and stat.S_ISDIR(info.st_mode):\\r\\n                    images[name] = str(path)\\r\\n            result[\\"candidates\\"].append({\\"root\\": str(directory), \\"files\\": files,\\r\\n                                         \\"missing\\": [name for name in MANIFEST_NAMES if name not in present],\\r\\n                                         \\"image_directory_candidates\\": images, \\"dicom_file_count\\": None})\\r\\n        if config.data_root is not None or depth >= config.max_depth:\\r\\n            continue\\r\\n        try:\\r\\n            with os.scandir(directory) as entries:\\r\\n                for entry in entries:\\r\\n                    if result[\\"entries_checked\\"] >= config.max_entries:\\r\\n                        result[\\"limit_reached\\"] = True\\r\\n                        break\\r\\n                    result[\\"entries_checked\\"] += 1\\r\\n                    if entry.name.startswith(\\".\\") or entry.name in IMAGE_DIRECTORIES or entry.name.casefold() in PROTECTED_COMPONENTS:\\r\\n                        continue\\r\\n                    child = directory / entry.name\\r\\n                    info = checked_stat(child)\\r\\n                    if info is not None and stat.S_ISDIR(info.st_mode):\\r\\n                        pending.append((child, depth + 1))\\r\\n        except OSError as error:\\r\\n            issue(directory, error)\\r\\n        if result[\\"limit_reached\\"]:\\r\\n            break\\r\\n    result[\\"scan_complete\\"] = cursor == len(pending) and not result[\\"limit_reached\\"] and not result[\\"error_count\\"]\\r\\n    result[\\"candidates\\"].sort(key=lambda item: item[\\"root\\"])\\r\\n    complete = [item for item in result[\\"candidates\\"] if not item[\\"missing\\"]]\\r\\n    if len(complete) > 1:\\r\\n        result[\\"status\\"] = \\"AMBIGUOUS_INPUT_ROOTS\\"\\r\\n    elif not result[\\"scan_complete\\"]:\\r\\n        result[\\"status\\"] = \\"DISCOVERY_INCOMPLETE\\"\\r\\n    elif complete:\\r\\n        result.update(status=\\"MANIFESTS_LOCATED\\", selected_root=complete[0][\\"root\\"])\\r\\n    elif result[\\"candidates\\"]:\\r\\n        result[\\"status\\"] = \\"INCOMPLETE_MANIFESTS\\"\\r\\n    return result\\r\\n\\r\\n\\r\\ndef read_csv_manifest(path: str | Path, config: AuditConfig) -> tuple[list[str], list[dict], str]:\\r\\n    source = safe_absolute(path)\\r\\n    base = safe_absolute(config.input_base)\\r\\n    if not source.is_relative_to(base) or source.name not in MANIFEST_NAMES:\\r\\n        raise InputError(\\"CSV source must be a named competition manifest within input_base\\")\\r\\n    if not stat.S_ISREG(source.lstat().st_mode):\\r\\n        raise InputError(\\"CSV source must be a regular file\\")\\r\\n    with source.open(\\"rb\\") as stream:\\r\\n        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):\\r\\n            raise InputError(\\"CSV source must be a regular file\\")\\r\\n        raw = stream.read(config.max_file_bytes + 1)\\r\\n    if len(raw) > config.max_file_bytes:\\r\\n        raise InputError(\\"CSV_FILE_LIMIT_EXCEEDED\\")\\r\\n    try:\\r\\n        reader = csv.DictReader(io.StringIO(raw.decode(\\"utf-8-sig\\"), newline=\\"\\"), strict=True)\\r\\n        columns = reader.fieldnames\\r\\n        if not columns or any(not name for name in columns) or len(set(columns)) != len(columns):\\r\\n            raise InputError(\\"CSV_HEADERS_MISSING_OR_DUPLICATED\\")\\r\\n        rows = []\\r\\n        for number, row in enumerate(reader, start=1):\\r\\n            if number > config.max_csv_rows:\\r\\n                raise InputError(\\"CSV_ROW_LIMIT_EXCEEDED\\")\\r\\n            if None in row or any(value is None for value in row.values()):\\r\\n                raise InputError(\\"CSV_ROW_WIDTH_INVALID\\")\\r\\n            rows.append(row)\\r\\n    except (UnicodeError, csv.Error) as error:\\r\\n        raise InputError(\\"CSV_ENCODING_OR_FORMAT_INVALID\\") from error\\r\\n    return columns, rows, hashlib.sha256(raw).hexdigest()\\r\\n","metrics":"\\"\\"\\"Observed-label ROC AUC with explicit support and undefined macro metrics.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport math\\r\\nfrom collections.abc import Iterable, Mapping, Sequence\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, parse_label, validate_identifier\\r\\nfrom .submission import validate_submission\\r\\n\\r\\n\\r\\ndef binary_auc(labels: Iterable[Any], scores: Iterable[Any]) -> float | None:\\r\\n    \\"\\"\\"Compute rank AUC, averaging tied ranks; return None for missing classes.\\"\\"\\"\\r\\n    observed_labels = list(labels)\\r\\n    observed_scores = list(scores)\\r\\n    if len(observed_labels) != len(observed_scores):\\r\\n        raise ContractError(\\"AUC labels and scores must have identical lengths\\")\\r\\n    pairs = []\\r\\n    for label, score in zip(observed_labels, observed_scores):\\r\\n        if not _numeric(label) or parse_label(label)[1] != \\"observed\\":\\r\\n            raise ContractError(\\"AUC requires observed numeric binary labels\\")\\r\\n        if not _numeric(score):\\r\\n            raise ContractError(\\"AUC requires numeric scores\\")\\r\\n        try:\\r\\n            numeric_score = float(score)\\r\\n        except (OverflowError, ValueError) as error:\\r\\n            raise ContractError(\\"AUC requires finite scores\\") from error\\r\\n        if not math.isfinite(numeric_score):\\r\\n            raise ContractError(\\"AUC requires finite scores\\")\\r\\n        pairs.append((numeric_score, float(label)))\\r\\n    positive_count = sum(label == 1.0 for _, label in pairs)\\r\\n    negative_count = len(pairs) - positive_count\\r\\n    if not positive_count or not negative_count:\\r\\n        return None\\r\\n    pairs.sort(key=lambda pair: pair[0])\\r\\n    positive_rank_sum = 0.0\\r\\n    start = 0\\r\\n    while start < len(pairs):\\r\\n        end = start + 1\\r\\n        while end < len(pairs) and pairs[end][0] == pairs[start][0]:\\r\\n            end += 1\\r\\n        mean_rank = (start + 1 + end) / 2.0\\r\\n        positive_rank_sum += mean_rank * sum(label == 1.0 for _, label in pairs[start:end])\\r\\n        start = end\\r\\n    return (positive_rank_sum - positive_count * (positive_count + 1) / 2.0) / (positive_count * negative_count)\\r\\n\\r\\n\\r\\ndef evaluate_predictions(\\r\\n    label_rows: Sequence[Mapping[str, Any]],\\r\\n    prediction_rows: Sequence[Mapping[str, Any]],\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Match studies exactly and evaluate each target on observed labels only.\\"\\"\\"\\r\\n    if not label_rows:\\r\\n        raise ContractError(\\"reference label table must not be empty\\")\\r\\n    references: dict[str, Mapping[str, Any]] = {}\\r\\n    for index, row in enumerate(label_rows):\\r\\n        if not isinstance(row, Mapping) or not set((STUDY_ID, *TARGETS)).issubset(row):\\r\\n            raise ContractError(f\\"reference row {index} is missing required cells\\")\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in references:\\r\\n            raise ContractError(f\\"duplicate reference identifier at row {index}\\")\\r\\n        references[uid] = row\\r\\n    validate_submission(prediction_rows, references)\\r\\n    predictions = {row[STUDY_ID]: row for row in prediction_rows}\\r\\n    per_target = {}\\r\\n    for target in TARGETS:\\r\\n        labels = []\\r\\n        scores = []\\r\\n        unknown_count = 0\\r\\n        for uid, row in references.items():\\r\\n            label, state = parse_label(row[target])\\r\\n            if state == \\"invalid\\":\\r\\n                raise ContractError(f\\"reference contains an invalid label for {target}\\")\\r\\n            if state == \\"unknown\\":\\r\\n                unknown_count += 1\\r\\n                continue\\r\\n            labels.append(label)\\r\\n            scores.append(predictions[uid][target])\\r\\n        positives = sum(label == 1.0 for label in labels)\\r\\n        per_target[target] = {\\r\\n            \\"observed\\": len(labels), \\"positive\\": positives,\\r\\n            \\"negative\\": len(labels) - positives, \\"unknown\\": unknown_count,\\r\\n            \\"auc\\": binary_auc(labels, scores),\\r\\n        }\\r\\n    defined = [stats[\\"auc\\"] for stats in per_target.values() if stats[\\"auc\\"] is not None]\\r\\n    complete = len(defined) == len(TARGETS)\\r\\n    return {\\r\\n        \\"status\\": \\"FULL_METRICS_AVAILABLE\\" if complete else \\"INCOMPLETE_METRICS\\",\\r\\n        \\"row_count\\": len(references), \\"per_target\\": per_target,\\r\\n        \\"macro_auc\\": sum(defined) / len(TARGETS) if complete else None,\\r\\n        \\"diagnostic_defined_target_macro_auc\\": sum(defined) / len(defined) if defined else None,\\r\\n        \\"defined_target_count\\": len(defined), \\"required_target_count\\": len(TARGETS),\\r\\n    }\\r\\n","submission":"\\"\\"\\"Validate the exact submission contract and write without overwriting files.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport csv\\r\\nimport math\\r\\nfrom collections.abc import Iterable, Mapping, Sequence\\r\\nfrom pathlib import Path\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, validate_identifier\\r\\nfrom .inputs import safe_absolute\\r\\n\\r\\nSUBMISSION_COLUMNS = (STUDY_ID, *TARGETS)\\r\\n\\r\\n\\r\\ndef validate_submission(\\r\\n    rows: Sequence[Mapping[str, Any]],\\r\\n    test_ids: Iterable[str],\\r\\n    columns: Sequence[str] | None = None,\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Require each test study once and twelve finite probabilities in order.\\"\\"\\"\\r\\n    expected_ids = [validate_identifier(uid) for uid in test_ids]\\r\\n    if not expected_ids:\\r\\n        raise ContractError(\\"test identifier set must not be empty\\")\\r\\n    if len(set(expected_ids)) != len(expected_ids):\\r\\n        raise ContractError(\\"test identifier list contains duplicates\\")\\r\\n    if not rows:\\r\\n        raise ContractError(\\"submission must contain at least one row\\")\\r\\n    if not isinstance(rows[0], Mapping):\\r\\n        raise ContractError(\\"submission row 0 must be a mapping\\")\\r\\n    actual_columns = tuple(columns) if columns is not None else tuple(rows[0])\\r\\n    if actual_columns != SUBMISSION_COLUMNS:\\r\\n        raise ContractError(\\"submission columns must exactly match the identifier and target order\\")\\r\\n    observed_ids: set[str] = set()\\r\\n    for index, row in enumerate(rows):\\r\\n        if not isinstance(row, Mapping) or set(row) != set(SUBMISSION_COLUMNS):\\r\\n            raise ContractError(f\\"submission row {index} has missing or extra cells\\")\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in observed_ids:\\r\\n            raise ContractError(f\\"duplicate submission identifier at row {index}\\")\\r\\n        observed_ids.add(uid)\\r\\n        for target in TARGETS:\\r\\n            value = row[target]\\r\\n            if not _numeric(value):\\r\\n                raise ContractError(f\\"submission row {index} has a nonnumeric probability for {target}\\")\\r\\n            try:\\r\\n                probability = float(value)\\r\\n            except (ValueError, OverflowError) as error:\\r\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\") from error\\r\\n            if not math.isfinite(probability) or not 0.0 <= probability <= 1.0:\\r\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\")\\r\\n    if observed_ids != set(expected_ids):\\r\\n        raise ContractError(\\"submission identifier set differs from the exact test identifier set\\")\\r\\n    return {\\r\\n        \\"status\\": \\"PASS\\", \\"row_count\\": len(rows), \\"unique_study_count\\": len(observed_ids),\\r\\n        \\"columns\\": list(SUBMISSION_COLUMNS), \\"probabilities_per_row\\": len(TARGETS),\\r\\n    }\\r\\n\\r\\n\\r\\ndef write_submission(\\r\\n    rows: Sequence[Mapping[str, Any]],\\r\\n    test_ids: Iterable[str],\\r\\n    path: str | Path,\\r\\n    columns: Sequence[str] = SUBMISSION_COLUMNS,\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Validate, reject protected/link paths, and refuse any existing output.\\"\\"\\"\\r\\n    validation = validate_submission(rows, test_ids, columns)\\r\\n    destination = safe_absolute(path)\\r\\n    with destination.open(\\"x\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\r\\n        writer = csv.DictWriter(stream, fieldnames=list(columns))\\r\\n        writer.writeheader()\\r\\n        writer.writerows(rows)\\r\\n    return {**validation, \\"path\\": str(destination), \\"size_bytes\\": destination.stat().st_size}\\r\\n"}}')
_RSNA_PACKAGE = 'rsna_knee_bundle'
for _module, _source in _RSNA_BUNDLE["sources"].items():
    _actual = hashlib.sha256(_source.encode("utf-8")).hexdigest()
    if _actual != _RSNA_BUNDLE["sha256"][_module + ".py"]:
        raise RuntimeError("Embedded source checksum mismatch: " + _module)
_RSNA_SOURCE_IDENTITY = {
    "kind": "embedded-notebook",
    "bundle_sha256": hashlib.sha256(json.dumps(_RSNA_BUNDLE, sort_keys=True, ensure_ascii=True, separators=(",", ":")).encode("utf-8")).hexdigest(),
    "source_sha256": dict(_RSNA_BUNDLE["sha256"]),
}

class _RsnaBundleLoader(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    package_name = _RSNA_PACKAGE

    def __init__(self, sources, fingerprint):
        # Bind this loader to its own bundle, not mutable notebook globals.
        self.sources = dict(sources)
        self.fingerprint = fingerprint

    def find_spec(self, fullname, path=None, target=None):
        if fullname == self.package_name:
            return importlib.util.spec_from_loader(fullname, self, is_package=True)
        prefix = self.package_name + "."
        if fullname.startswith(prefix):
            name = fullname[len(prefix):]
            if name in self.sources and name != "__init__":
                return importlib.util.spec_from_loader(fullname, self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        name = "__init__" if module.__name__ == self.package_name else module.__name__.split(".")[-1]
        module.__file__ = "<embedded:" + name + ".py>"
        module.__rsna_bundle_sha256__ = self.fingerprint
        code = compile(self.sources[name], module.__file__, "exec")
        exec(code, module.__dict__)

# Preserve class identity when re-running an identical, verified bundle.
# Changed sources are reloaded and require the configuration cell to run again.
_rsna_fingerprint = _RSNA_SOURCE_IDENTITY["bundle_sha256"]
_rsna_cached = {
    name: module for name, module in sys.modules.items()
    if name == _RSNA_PACKAGE or name.startswith(_RSNA_PACKAGE + ".")
}
_rsna_same_bundle = (
    _RSNA_PACKAGE in _rsna_cached
    and all(getattr(module, "__rsna_bundle_sha256__", None) == _rsna_fingerprint
            for module in _rsna_cached.values())
)
if not _rsna_same_bundle:
    for _name in _rsna_cached:
        del sys.modules[_name]
    sys.meta_path[:] = [finder for finder in sys.meta_path if getattr(finder, "package_name", None) != _RSNA_PACKAGE]
    sys.meta_path.insert(0, _RsnaBundleLoader(_RSNA_BUNDLE["sources"], _rsna_fingerprint))
from rsna_knee_bundle.config import AuditConfig
from rsna_knee_bundle.audit import run_preflight, write_report


In [ ]:
# Generated configuration. Use the project builder to change settings.
CONFIG = {'input_base': '/kaggle/input',
 'inspect_csv': False,
 'output_dir': '/kaggle/working/rsna_audit'}
audit_config = AuditConfig.from_dict(CONFIG)


In [ ]:
if 'audit_config' not in globals() or not isinstance(audit_config, AuditConfig):
    raise RuntimeError('Missing or stale AuditConfig: rerun the configuration cell before the audit, or use Run All.')
audit_report = run_preflight(audit_config, source_identity=_RSNA_SOURCE_IDENTITY)
print(json.dumps(audit_report, sort_keys=True, indent=2))


In [ ]:
if 'audit_report' not in globals() or 'audit_config' not in globals() or not isinstance(audit_config, AuditConfig):
    raise RuntimeError('Run the configuration and audit cells first, or use Run All.')
if audit_report.get('source_identity') != _RSNA_SOURCE_IDENTITY or audit_report.get('configuration') != audit_config.to_dict():
    raise RuntimeError('The audit report is stale; run the configuration and audit cells again, or use Run All.')
report_path = write_report(audit_report, audit_config)
print(str(report_path))
